# 🛠️ 25 · Restoration and Inverse Problems
### Recovering a clean signal from a known degradation — from a catastrophic inverse filter to Wiener filtering to a learned, content-aware restorer

**Part H · Restoring signal** · [⟵ 24 Denoising & super-resolution](24_denoising_and_super_resolution.ipynb) · [🏠 Course home](../README.md) · [26 Colocalization & spatial statistics ⟶](26_colocalization_and_spatial_statistics.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/25_restoration_and_inverse_problems.ipynb)

| | |
|---|---|
| **Hardware target** | CPU; the one small CNN training takes roughly 20-40 seconds |
| **Core frameworks** | NumPy, SciPy (FFT), PyTorch (one tiny CNN, CPU-only), Matplotlib |
| **Prerequisites** | Module 11 (filtering & point operations), Module 12 (frequency domain), Module 24 (denoising, PSNR/SSIM) |
| **Estimated time** | 70-90 minutes |
| **02-740 topics covered** | the degradation model g = h*f + n, restoration vs. enhancement, inverse filtering and its frequency-domain instability, Wiener (minimum mean-square-error) filtering, content-aware restoration for fluorescence microscopy (CARE) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All images, degradations and training data in this notebook are SIMULATED by code — nothing is downloaded.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- state the degradation model $g = h*f + n$ and explain why restoration is a formally-posed inverse problem, in contrast with enhancement, which has no explicit, invertible degradation model;
- implement naive inverse filtering in the frequency domain and explain precisely why it is numerically unstable wherever the degradation's frequency response is small;
- implement Wiener (minimum mean-square-error) filtering and explain its regularization parameter as a noise-to-signal trade-off — including when mis-tuning it, or applying it to an image that was never severely degraded, can leave you worse off than doing nothing at all;
- train a small CARE-style convolutional network that learns a restoration mapping directly from paired degraded/clean data, and quantitatively compare it (PSNR/SSIM) against the hand-derived classical filters;
- explain how the inverse-problem framing generalizes far beyond spatial blur (cross-referencing colour/stain unmixing from Module 11) and how modern restoration methods extend the same idea with learned priors.

## 🗺️ Contents

1. The degradation model: restoration vs. enhancement
2. Naive inverse filtering — and why it explodes
3. Wiener filtering: a regularized inverse, and its bias–variance trade-off
4. Learned, content-aware restoration (CARE-style)
5. Beyond blur: other degradation models and modern inverse-problem solvers

## 🧫 The biological question

A microscope image is never the "true" specimen directly — out-of-focus light, diffraction, sensor and motion blur, and noise all intervene between the sample and the pixels you actually get. Module 24 treated the noise half of that story. This notebook adds the other half, blur, and asks a sharper question than "make it look nicer": given a corrupted observation and some knowledge of *how* it was corrupted, how much of the original signal can we actually, provably recover — and when does attempting to recover it make things worse instead of better? Framing restoration this way, as an estimation problem with an explicit forward (degradation) model, is what separates it from ad hoc visual enhancement, and is the same framing used throughout modern computational imaging.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
import torch
import torch.nn as nn
SEED = 17
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

We reuse `simulate_nuclei_image`, `degrade_image` (the forward degradation model $g=h*f+n$, with a known point-spread function), `wiener_deconvolve`, `psnr`, `simple_ssim`, and `show_image_grid`.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def degrade_image(image: np.ndarray, psf_sigma: float = 1.5, noise_sigma: float = 0.04,
                   seed: int = 0) -> dict:
    """Blur + noise a clean image with a KNOWN point-spread function (the forward model of
    restoration/deconvolution). Returns ``degraded`` and the ``psf`` used."""
    rng = np.random.default_rng(seed)
    psf = gaussian_psf(psf_sigma)
    degraded = blur_image(image, psf)
    degraded = add_gaussian_noise(degraded, noise_sigma, rng).clip(0, None)
    return dict(degraded=degraded, psf=psf, psf_sigma=psf_sigma, noise_sigma=noise_sigma)


def wiener_deconvolve(degraded: np.ndarray, psf: np.ndarray, noise_to_signal: float = 0.01) -> np.ndarray:
    """Wiener deconvolution: inverse-filter a blurred, noisy image given its (known) PSF and an
    estimated noise-to-signal power ratio, which regularises the inverse near-zero frequencies."""
    h, w = degraded.shape
    psf_padded = np.zeros((h, w))
    ph, pw = psf.shape
    psf_padded[:ph, :pw] = psf
    psf_padded = np.roll(psf_padded, (-ph // 2, -pw // 2), axis=(0, 1))
    H = np.fft.fft2(psf_padded)
    G = np.fft.fft2(degraded)
    Hc = np.conj(H)
    wiener = Hc / (H * Hc + noise_to_signal)
    return np.real(np.fft.ifft2(wiener * G))


def psnr(reference: np.ndarray, estimate: np.ndarray, data_range: float | None = None) -> float:
    """Peak signal-to-noise ratio (dB) between a reference and an estimate; higher is better."""
    data_range = data_range or (reference.max() - reference.min() + 1e-9)
    mse = np.mean((reference - estimate) ** 2)
    if mse <= 1e-12:
        return float("inf")
    return 10 * np.log10(data_range ** 2 / mse)


def simple_ssim(a: np.ndarray, b: np.ndarray, data_range: float | None = None) -> float:
    """A simplified, global (whole-image, not windowed) structural-similarity index: combines
    luminance, contrast and structure agreement; 1.0 = identical. For a full windowed SSIM use
    ``skimage.metrics.structural_similarity``."""
    data_range = data_range or (max(a.max(), b.max()) - min(a.min(), b.min()) + 1e-9)
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    mu_a, mu_b = a.mean(), b.mean()
    var_a, var_b = a.var(), b.var()
    cov = ((a - mu_a) * (b - mu_b)).mean()
    return float(((2 * mu_a * mu_b + c1) * (2 * cov + c2)) /
                 ((mu_a ** 2 + mu_b ** 2 + c1) * (var_a + var_b + c2)))


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1 · The degradation model: restoration vs. enhancement

The standard model for a degraded image is

$$g(x,y) = h(x,y) * f(x,y) + \eta(x,y) \qquad\Longleftrightarrow\qquad G(u,v) = H(u,v)\,F(u,v) + N(u,v)$$

where $f$ is the true, undegraded image, $h$ is a (here: assumed *known*) degradation function — for optical blur, the point-spread function (PSF) of the imaging system — $\eta$ is additive noise, and $g$ is what the detector actually records. Convolution in space becomes multiplication in frequency, which is exactly why frequency-domain reasoning (Module 12) is the natural setting for restoration.

**Restoration** takes this model literally: given $g$ and some knowledge of $h$ and the statistics of $\eta$, estimate $f$ by inverting the degradation. Because the problem is posed with an explicit, quantifiable forward model, restoration quality can be measured precisely (Section 3 onward does exactly this with PSNR/SSIM). **Enhancement** — contrast stretching, unsharp masking, histogram equalization (Module 11) — has no such model: it reshapes the image to look better or to emphasize chosen features, without any claim to be undoing a specific physical degradation. Both are useful; only one of them is answering "what did the original signal actually look like?".

In [ ]:
# ── 1.1 Two point-spread functions of different severity ──
psf_mild = gaussian_psf(sigma=1.5)
psf_severe = gaussian_psf(sigma=3.5)
show_image_grid([psf_mild, psf_severe],
                 titles=["mild PSF (sigma=1.5 px)", "severe PSF (sigma=3.5 px)"])

🔍 **What just happened?** Both kernels are normalised, smooth bumps a handful of pixels wide — this is what a diffraction-limited microscope's PSF typically looks like. A wider PSF spreads each point of true signal over more neighbouring pixels, destroying more fine detail — and, as Sections 2–3 will show, making the degradation harder to recover *and* more worthwhile to attempt.

In [ ]:
# ── 1.2 Degrade a clean scene with the severe PSF plus noise ──
clean_scene = simulate_nuclei_image(n_cells=14, size=128, noise_sigma=0.0, seed=SEED)["image"]
degraded_result = degrade_image(clean_scene, psf_sigma=3.5, noise_sigma=0.02, seed=SEED)
degraded, psf = degraded_result["degraded"], degraded_result["psf"]

show_image_grid([clean_scene, degraded],
                 titles=["clean f(x,y)", "degraded g(x,y) = h*f + noise"])
print(f"degraded vs clean:  PSNR = {psnr(clean_scene, degraded):.2f} dB   SSIM = {simple_ssim(clean_scene, degraded):.3f}")

🧪 **Ground-truth check.** `degrade_image` is our forward model with full knowledge of the ground truth: we know the exact `psf` used and the exact `noise_sigma`, which is precisely the "known degradation process" restoration assumes. In a real experiment the PSF is instead measured (e.g. imaging sub-resolution beads) or estimated, and the noise level is estimated from the sensor — restoration quality degrades gracefully as that knowledge gets less exact, but the method below still needs *some* estimate of both to work at all.

### 📘 Lecture notes — restoration vs. enhancement, and where degradation comes from

Restoration is designed to reduce noise and recover resolution lost to a *specific, modelled* degradation process; enhancement emphasises or extracts features for a viewer without necessarily producing a scientifically accurate reconstruction. This is why restoration problems can be quantified precisely (we can ask "how close is our estimate to the true $f$?" and get a number) while enhancement criteria are comparatively subjective. In bioimaging specifically, degradation has several common physical sources working together: optical aberrations and defocus, sensor blur from spatial averaging at each photosite, motion blur from live-sample or stage drift, and noise from both the photon-counting (shot) process and the sensor's own read-out electronics. Module 11 already covers a *different* restoration problem with the same logic — colour/stain unmixing for brightfield histology (Ruifrok & Johnston, 2001) inverts a known *linear mixing* degradation model (several dyes' absorbance spectra summed together) rather than a spatial PSF. Section 5 returns to that connection explicitly.

✏️ **Your turn.** Re-run cell 1.2 with `psf_sigma=1.5` (the mild PSF from 1.1) instead of `3.5`, keeping `noise_sigma=0.02`. How much does the degraded-vs-clean PSNR change compared to the severe-blur case? Keep this milder scene in mind — Section 3 revisits exactly this comparison.

## 2 · Naive inverse filtering — and why it explodes

If $g = h*f + \eta$, the most direct idea is to divide out $H$ in the frequency domain:

$$\hat{F}(u,v) = \frac{G(u,v)}{H(u,v)} = F(u,v) + \frac{N(u,v)}{H(u,v)}$$

This is an interesting expression: even with $H$ known exactly, $f$ cannot be recovered exactly, because $N(u,v)$ is unknown — and wherever $H(u,v)$ has a small magnitude, the ratio $N(u,v)/H(u,v)$ can dominate the whole estimate. A Gaussian PSF's frequency response is itself a (decaying) Gaussian, so it gets arbitrarily close to zero at high spatial frequencies — exactly where this blow-up bites hardest.

In [ ]:
# ── 2.1 The naive inverse filter: F_hat = G / H ──
def naive_inverse_filter(degraded, psf, eps=1e-3):
    """Undo a known blur by dividing out its frequency response. ``eps`` only guards
    against literal division by zero — it is NOT a principled way to handle noise
    (Section 3 does that properly)."""
    h, w = degraded.shape
    psf_padded = np.zeros((h, w))
    ph, pw = psf.shape
    psf_padded[:ph, :pw] = psf
    psf_padded = np.roll(psf_padded, (-ph // 2, -pw // 2), axis=(0, 1))
    H = np.fft.fft2(psf_padded)
    G = np.fft.fft2(degraded)
    F_hat = G / (H + eps)
    return np.real(np.fft.ifft2(F_hat))

inverse_restored = naive_inverse_filter(degraded, psf)
show_image_grid([clean_scene, degraded, np.clip(inverse_restored, 0, 1)],
                 titles=["clean", "degraded", "naive inverse filter"])
print(f"degraded:             PSNR = {psnr(clean_scene, degraded):6.2f} dB   SSIM = {simple_ssim(clean_scene, degraded):.3f}")
print(f"naive inverse filter: PSNR = {psnr(clean_scene, inverse_restored):6.2f} dB   SSIM = {simple_ssim(clean_scene, inverse_restored):.3f}")

🔍 **What just happened?** The "restored" image is overwhelmed by amplified noise — visually closer to static than to the clean scene, and its PSNR is strongly *negative* (far worse than the degraded image's own PSNR, i.e. worse than not restoring at all). This is exactly the instability flagged above: dividing by a near-zero $H(u,v)$ turns a small amount of noise at that frequency into an enormous error.

In [ ]:
# ── 🧪 Ground-truth check ──
assert psnr(clean_scene, inverse_restored) < 0 < psnr(clean_scene, degraded), \
    "the naive inverse filter should be catastrophically worse than not restoring at all"
print("Confirmed: naive inverse filtering is far worse than doing nothing on this degraded image.")

✏️ **Your turn.** Try `naive_inverse_filter(degraded, psf, eps=1e-2)` and `eps=3e-1`, much larger than the default `1e-3`. Does a bigger safety floor fix the instability, or only dampen it? (Check the PSNR at each setting.) Either way, `eps` was chosen purely to avoid dividing by exactly zero — it encodes no information about how much noise is actually present in `degraded`, which is exactly what Section 3's Wiener filter adds.

## 3 · Wiener filtering: a regularized inverse, and its bias–variance trade-off

The Wiener (minimum mean-square-error) filter incorporates both the degradation function *and* the statistical characteristics of the noise into the restoration, rather than ignoring the noise entirely as the naive inverse filter does. Treating image and noise as random processes and minimising the expected squared error between the estimate and the true $f$, assuming they are uncorrelated, gives a closed-form solution in the frequency domain:

$$\hat{F}(u,v) = \left[\frac{H^*(u,v)}{|H(u,v)|^2 + K}\right] G(u,v)$$

where $K$ is an assumed **noise-to-signal power ratio**. As $K \to 0$, this expression reduces to the naive inverse filter ($H^*/|H|^2 = 1/H$); as $K \to \infty$, the filter suppresses the output almost entirely. $K$ is therefore not a free dial to be cranked up for safety — it is a genuine bias–variance trade-off: too small, and noise is amplified just as badly as Section 2; too large, and the blur is barely undone at all.

In [ ]:
# ── 3.1 Sweep the regularization parameter K (noise_to_signal) ──
k_values = np.geomspace(5e-4, 2e-1, 15)
wiener_psnrs = [psnr(clean_scene, wiener_deconvolve(degraded, psf, noise_to_signal=k)) for k in k_values]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(k_values, wiener_psnrs, "o-", label="Wiener filter")
ax.axhline(psnr(clean_scene, degraded), color="gray", linestyle="--", label="degraded (do nothing)")
ax.set_xscale("log")
ax.set_xlabel("noise_to_signal (K)")
ax.set_ylabel("PSNR (dB) vs. clean")
ax.set_title("Wiener filter quality vs. regularization strength")
ax.legend()
plt.show()

best_k = k_values[int(np.argmax(wiener_psnrs))]
best_wiener = wiener_deconvolve(degraded, psf, noise_to_signal=best_k)
print(f"best K = {best_k:.4f}  ->  PSNR = {psnr(clean_scene, best_wiener):.2f} dB   SSIM = {simple_ssim(clean_scene, best_wiener):.3f}")
print(f"degraded (do nothing):  PSNR = {psnr(clean_scene, degraded):.2f} dB   SSIM = {simple_ssim(clean_scene, degraded):.3f}")

🔍 **What just happened?** The PSNR-vs-$K$ curve is U-shaped. With $K$ too small, the Wiener filter behaves like Section 2's unstable naive inverse filter (noise dominates); with $K$ too large, it barely deconvolves anything and under-corrects. Somewhere in between — found here purely by sweeping against the (simulated, known) ground truth — Wiener filtering genuinely beats doing nothing at all on this *severely* blurred scene (`psf_sigma=3.5`). This is the same bias–variance pattern that shows up throughout imaging: more aggressive restoration trades residual bias (leftover blur) for added variance (amplified noise), and the regularization strength is the knob that sets where you land on that trade.

In [ ]:
# ── 🧪 Ground-truth check ──
assert max(wiener_psnrs) > psnr(clean_scene, degraded), \
    "a well-tuned Wiener filter should beat doing nothing on this severely blurred image"
print(f"Confirmed: the best swept K beats doing nothing by {max(wiener_psnrs) - psnr(clean_scene, degraded):.2f} dB.")

### 📘 Lecture notes — Wiener filtering is a real improvement, but not a free one

Side-by-side comparisons of the inverse filter and the Wiener filter consistently show the Wiener result contains far less noise and performs significantly better. That comparison, however, is usually shown on a scene that is blurred enough for restoration to be worth the risk. If you repeat Section 1.2 with the *mild* PSF (`psf_sigma=1.5`) instead, the picture changes: a weak blur barely raises the pixelwise error to begin with, so there is little to gain from undoing it, while Wiener filtering still injects some frequency-dependent noise amplification whether or not that trade is worth making. On a mildly blurred image, even a well-tuned Wiener filter can end up with *worse* PSNR/SSIM than simply leaving the image alone — regularization strength is not the only thing that has to be matched to the data; so does the decision to restore at all.

✏️ **Your turn.** Repeat cell 3.1 on the *mild*-PSF scene from the previous ✏️ Your turn (`psf_sigma=1.5`, `noise_sigma=0.02`). Does any swept $K$ give a Wiener PSNR that beats the (new) degraded image? What does this tell you about when deconvolution is, and is not, worth attempting?

## 4 · Learned, content-aware restoration (CARE-style)

*Content-aware image restoration: pushing the limits of fluorescence microscopy* (Weigert et al., 2018) replaces the hand-derived Wiener filter with a convolutional neural network trained directly on paired data: registered low-signal-to-noise-ratio and high-SNR acquisitions of the same sample. The network (in the original paper, a U-Net-scale architecture with roughly $10^6$ parameters, trained with a per-pixel similarity loss) learns the restoration mapping end to end, rather than having it derived analytically from an assumed noise model — and the paper shows this consistently and substantially outperforms classical baselines, enabling usable imaging at laser power and photon budgets that would otherwise be too noisy to work with at all (crucial for live-cell imaging, where phototoxicity limits how much light you can use).

Below is a toy analogue in the same spirit as Module 24's denoising autoencoder: a small CNN trained on *simulated* degraded/clean pairs (the same `degrade_image` forward model from Section 1), evaluated on a held-out scene never seen in training.

In [ ]:
# ── 4.1 A small restoration CNN, trained from scratch on paired degraded/clean scenes ──
class TinyCARE(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)

def make_restoration_pair(seed, size=64, n_cells=8, psf_sigma=3.5, noise_sigma=0.02):
    clean_img = simulate_nuclei_image(n_cells=n_cells, size=size, noise_sigma=0.0, seed=seed)["image"]
    clean_img = clean_img.astype(np.float32)
    degraded_img = degrade_image(clean_img, psf_sigma=psf_sigma, noise_sigma=noise_sigma,
                                  seed=seed + 777)["degraded"].astype(np.float32)
    return degraded_img, clean_img

train_pairs = [make_restoration_pair(seed=100 + i) for i in range(12)]
X_train = torch.tensor(np.stack([p[0] for p in train_pairs]))[:, None, :, :]
Y_train = torch.tensor(np.stack([p[1] for p in train_pairs]))[:, None, :, :]

torch.manual_seed(SEED)
care_net = TinyCARE()
optimizer = torch.optim.Adam(care_net.parameters(), lr=1.5e-2)
loss_fn = nn.MSELoss()
for epoch in range(200):
    optimizer.zero_grad()
    loss = loss_fn(care_net(X_train), Y_train)
    loss.backward()
    optimizer.step()
print(f"final training MSE: {loss.item():.5f}")

# evaluate on a held-out scene (different seed, never seen during training)
test_degraded, test_clean = make_restoration_pair(seed=999)
with torch.no_grad():
    care_restored = care_net(torch.tensor(test_degraded)[None, None]).numpy()[0, 0]

🧪 **Ground-truth check.** `test_degraded`/`test_clean` use `seed=999`, which never appeared among the 12 training seeds (100–111) — a genuine held-out evaluation, exactly as in Module 24.

In [ ]:
# ── 4.2 Compare degraded, Wiener (reusing Section 3's swept K), and the learned restorer ──
test_wiener = wiener_deconvolve(test_degraded, degrade_image(test_clean, psf_sigma=3.5, noise_sigma=0.02,
                                                              seed=999 + 777)["psf"], noise_to_signal=best_k)

show_image_grid([test_clean, test_degraded, test_wiener, care_restored],
                 titles=["clean (held out)", "degraded", f"Wiener (K={best_k:.3f})", "TinyCARE (learned)"])

for name, estimate in [("degraded (do nothing)", test_degraded), ("Wiener filter", test_wiener),
                       ("TinyCARE (learned)", care_restored)]:
    print(f"{name:24s}  PSNR = {psnr(test_clean, estimate):5.2f} dB   SSIM = {simple_ssim(test_clean, estimate):.3f}")

🔍 **What just happened?** Expect the learned restorer to come out ahead of the hand-derived Wiener filter here: Wiener filtering applies one global, frequency-domain correction derived from a simplified linear-Gaussian noise assumption, while the CNN is fit directly to minimise reconstruction error on example pairs and can exploit nonlinear, spatially-varying, class-specific structure (e.g. "a smooth round blob edge, not noise") that no single hand-derived filter captures. This is the same classical-vs-learned trade-off Module 24 found for denoising — now applied to the harder joint blur-plus-noise problem, and it is precisely CARE's premise: learn the restoration mapping instead of deriving it analytically.

### 📘 Lecture notes — what this toy demo leaves out

Our `TinyCARE` network is a simplified analogy, not a reproduction of CARE: it is a plain stack of convolutions (not a U-Net), trained on a single, 2-D, single-channel simulated blur-plus-noise degradation (not registered real low-SNR/high-SNR volume pairs), with far fewer parameters than the original's roughly $10^6$. What carries over is the core idea the paper demonstrated at scale: given enough well-registered paired examples, a trained network can out-perform classical restoration baselines (the paper compares against non-local means) by quantitative measures like PSNR/SSIM, closely approaching ground truth — and can do so even when the input SNR is very low, which is exactly the live-imaging regime where classical filtering struggles most.

✏️ **Your turn.** Reduce the training set in cell 4.1 from 12 pairs to 4 (e.g. `range(4)` instead of `range(12)`) and retrain. How much of TinyCARE's advantage over the Wiener filter survives with that much less training data? What does this suggest about the real cost of a CARE-style learned restorer compared to a classical one?

## 5 · Beyond blur: other degradation models and modern inverse-problem solvers

Nothing about the restoration framing in this notebook is specific to spatial blur. Module 11 covers a *different* restoration problem built on exactly the same logic: colour/stain unmixing for brightfield histology (Ruifrok & Johnston, 2001) treats the observed RGB image as a *known, linear mixture* of a small number of dye absorbance spectra (e.g. haematoxylin and eosin), and restoration means inverting that mixing matrix rather than inverting a 2-D point-spread function. Same forward model logic — $g$ = (known degradation) applied to $f$, plus noise — completely different physics.

More generally, both filters built in this notebook are special cases of a much broader strategy: pose restoration as minimising a *data-fit term* (how well does my estimate, degraded, match what I actually observed?) plus a *regularizer/prior* (how plausible is this estimate, independent of the data?), then solve that optimisation iteratively. The Wiener filter's single global $K$ is the simplest possible prior (a flat power-spectrum assumption); classical methods like half-quadratic splitting and ADMM solve the same style of problem with hand-designed priors such as total variation; and increasingly, the prior itself is *learned* — a pretrained denoiser, or a generative model, plugged into the same iterative solver as a so-called "plug-and-play" prior. Diffusion models (Module 27) are a particularly active example of this last idea: a diffusion model trained to generate realistic images can be repurposed, without retraining, as a learned prior for restoring degraded ones — the same data-fit-plus-prior logic this notebook built from scratch, extended with a far richer prior than any single hyperparameter can express.

## 📌 Key takeaways

- Restoration formalises the degradation model $g = h*f + n$ and estimates $f$ by inverting it; enhancement has no such explicit, invertible model, which is why restoration quality can be quantified precisely and enhancement quality cannot.
- Naive inverse filtering ($\hat{F} = G/H$) is numerically unstable wherever $H(u,v)$ is small — it can amplify noise catastrophically, far worse than not restoring at all, and a larger "safety" epsilon only dampens this without fixing it.
- Wiener filtering regularizes the inverse with a noise-to-signal parameter $K$; mis-tuning $K$, or applying deconvolution to a mildly-degraded image in the first place, can still leave you worse off than doing nothing — restoration is a genuine bias–variance trade-off, not an automatic win.
- A small CNN trained end to end on paired degraded/clean data (a toy analogue of CARE) can outperform the hand-derived Wiener filter by learning image-class-specific structure, at the cost of needing training data and compute — the same classical-vs-learned pattern Module 24 found for denoising.
- The inverse-problem framing generalises far beyond spatial blur (colour/stain unmixing in Module 11 is a restoration problem with a completely different degradation model), and modern solvers extend the same data-fit-plus-prior idea with learned priors, including diffusion models (Module 27).

## ✅ Checkpoint

1. Why does naive inverse filtering explode at frequencies where $|H(u,v)|$ is small, and why doesn't a tiny `eps` fix it?
   <details><summary>Answer</summary>Because $\hat{F} = G/H = F + N/H$, wherever $H$ is near zero any nonzero noise $N$ at that frequency gets divided by an arbitrarily small number and dominates the estimate; a tiny `eps` only avoids literal division by zero, it encodes no information about how much noise is actually present, so it barely limits the amplification.</details>
2. What role does the noise-to-signal parameter $K$ play in the Wiener filter, and what happens at the two extremes $K\to 0$ and $K\to\infty$?
   <details><summary>Answer</summary>$K$ encodes an assumed noise-to-signal power ratio; as $K\to 0$ the filter reduces to the unstable naive inverse filter (since $H^*/(|H|^2+K) \to 1/H$), and as $K\to\infty$ the filter's output is suppressed almost to zero (heavily over-regularized), so $K$ trades noise amplification against under-correction.</details>
3. Why might Wiener-filtering a mildly-blurred image make its PSNR/SSIM *worse* rather than better?
   <details><summary>Answer</summary>If the blur barely raises the pixelwise error to begin with, there is little error left to recover, while any deconvolution still amplifies some noise at every frequency; the small potential gain from undoing a weak blur can be outweighed by the nontrivial cost of amplified noise, so with mild blur the best action can be to leave the image alone.</details>
4. In what precise sense is "learn the restoration mapping end to end from paired data" (Section 4, CARE) a different strategy from "derive a closed-form filter from an assumed noise model" (Section 3, Wiener), even though both start from the same degradation model?
   <details><summary>Answer</summary>The Wiener filter is derived analytically from a single global noise-to-signal statistic and applied identically at every frequency, whereas the learned network is fit directly to minimise reconstruction error on example pairs and can exploit nonlinear, spatially-varying, class-specific structure that no single hand-derived filter captures.</details>

## 🧠 Questions & discussion

- Section 3 found the best $K$ by sweeping against the clean ground truth — a luxury available only because we simulated the data ourselves. In a real experiment you never see the clean image. Sketch an approach to estimating or choosing $K$ without it.
- CARE trains on real, registered pairs of low-SNR and high-SNR acquisitions of the *same* sample. What would go wrong if those two acquisitions were not well spatially aligned with each other?
- Section 5 mentioned "plug-and-play" learned priors (a pretrained denoiser, or a diffusion model) used inside classical optimisation loops. What property would such a learned prior need, to be reusable across genuinely different degradation models (blur removal *and* stain unmixing), rather than needing to be retrained separately for each one?

## 🚀 Bonus — going further

- Increase `noise_sigma` to `0.05` in Section 1 (keeping `psf_sigma=3.5`) and re-run Sections 2–4. Does the gap between the naive inverse filter's catastrophe and the Wiener filter's recovery widen or narrow?
- Instead of treating $K$ as a single free hyperparameter swept against ground truth, estimate it directly from a small, nominally-flat background patch of `degraded` (its local variance is an estimate of the noise power) and compare the resulting $K$ to the one Section 3 found by sweeping.
- Retrain `TinyCARE` with only 4 training pairs instead of 12 (Section 4's ✏️ Your turn) and plot how its PSNR advantage over the Wiener filter shrinks as a function of training-set size.

## 📝 Lab note

```text
Module 25 — Restoration and inverse problems
Data: SIMULATED nuclei image, course_toolkit.degrade_image(psf_sigma=3.5, noise_sigma=0.02, seed=17)

Section 2 (naive inverse filter):
  degraded              PSNR [ ] dB  SSIM [ ]
  naive inverse filter  PSNR [ ] dB  SSIM [ ]

Section 3 (Wiener filter sweep):
  best K [ ]
  best Wiener           PSNR [ ] dB  SSIM [ ]

Section 4 (held-out scene, seed=999):
  degraded              PSNR [ ] dB  SSIM [ ]
  Wiener (best K)       PSNR [ ] dB  SSIM [ ]
  TinyCARE (learned)    PSNR [ ] dB  SSIM [ ]

Notes:
```

## 📚 References

1. Gonzalez, R. C., & Woods, R. E. *Digital Image Processing* — image restoration: the degradation model, inverse filtering, and Wiener (minimum mean-square-error) filtering.
2. Wiener, N. (1949). *Extrapolation, Interpolation, and Smoothing of Stationary Time Series*. MIT Press.
3. Weigert, M. et al. (2018). Content-aware image restoration: pushing the limits of fluorescence microscopy. *Nature Methods*.
4. Ruifrok, A. C., & Johnston, D. A. (2001). Quantification of histochemical staining by color deconvolution. *Analytical and Quantitative Cytology and Histology*.
5. Afonso, M. V., Bioucas-Dias, J. M., & Figueiredo, M. A. T. (2010). Fast image recovery using variable splitting and constrained optimization. *IEEE Transactions on Image Processing*.
6. Kawar, B., Elad, M., Ermon, S., & Song, J. (2022). Denoising diffusion restoration models. *NeurIPS*.

## ➡️ Next

[**Module 26 — Colocalization and spatial statistics**](26_colocalization_and_spatial_statistics.ipynb): from recovering a single clean image, we move to asking quantitative questions about relationships *within* an image — whether two labelled channels genuinely overlap, and whether a population of detected objects is clustered, dispersed, or consistent with pure chance.